# Task 01: Introduction to PyTorch

## Introduction

The goal of this task is to get us thinking not just about training models, but about our *training pipelines*.

A neural network is a function, $f$, that accepts data inputs, $\boldsymbol{X}$, and weights, $\boldsymbol{\Theta}$ that produces predictions $\boldsymbol{\hat{y}}$,

$$
\boldsymbol{\hat{y}} = f(\boldsymbol{\Theta}; \boldsymbol{X}).
$$

Meanwhile, a neural network training process is itself a function, $g$, which accepts as input a dataset $\boldsymbol{X}$, and for supervised algorithms a set of targets $y$, along with a set of training configuration hyperparameters $\boldsymbol{\Omega}$ which define how the process is performed, and produces as output the weights of a neural network, $\boldsymbol{\Theta}$,

$$
\boldsymbol{\Theta} = g(\boldsymbol{\Omega}; \boldsymbol{X}, \boldsymbol{y}).
$$

It is helpful to think of the training function, $g$, as a pipeline, composed of several training steps, which can include preprocessing, post-processing, etc.

$$
g = g_N \circ\ \cdots\ \circ g_1.
$$

For example, $g_1$ might be a preprocessing step, then $g_2$ might be a training step, and $g_3$ might be a pruning step in a basic pipeline where data $(\boldsymbol{X}, \boldsymbol{y})$ goes in and weights $\boldsymbol{\Theta}$ come out.

Generally, we separate out a training, validation, and testing set. For MNIST, there is an official test set, so we split the training set into training and validation portions. The training set is used to estimate $\boldsymbol{\Theta}$, the parameters of the network. The validation set is used to choose $\boldsymbol{\Omega}$, the hyperparameters of the training process itself. The test data is held out entirely to estimate final performance.

We will learn to think of the training process this way by modifying some example code for a basic MNIST classification task. We begin with some imports.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms


## Task 01 - Part 1

Your first task is to:

* Add layer definitions to the following neural network class
* Define the forward pass

You can find starting architectures online. It is important to know there is not a complete theory to identify a best architecture before starting the problem. Trial and error (by iterative training and experimentation) is generally required to prove or disprove the utility of an architecture.

Recall some of our intuition about the way linear and nonlinear transforms work. Chaining linear transforms together simplifies into another linear transform which does not increase the space of functions our network can support. The space of functions a network can support is also called the *hypothesis space*. The number of layers, width of those layers, and types of transformations employed are all therefore architecture design choices we can explore experimentally to find a suitable model for our task from the space of possible models we could have chosen.

**IMPORTANT:** The starter code assumes that your network accepts an MNIST image as input and returns a tensor containing 10 output values for each image, one for each digit class (0 through 9).


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        raise NotImplementedError("Not implemented!")
        # TODO: Define the layers of the model here

    def forward(self, x):
        raise NotImplementedError("Not implemented!")
        # TODO: Implement the forward pass of the model here
        # Uncomment the return once implemented
        # return output


def run_training_epoch(
    training_params, model, device, train_loader, optimizer, epoch
):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % training_params.log_interval == 0:
            print(
                f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100.0 * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}"
            )
            if training_params.dry_run:
                break


def evaluate(model, device, data_loader, dataset_name="Test"):
    model.eval()
    loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss += F.cross_entropy(
                output, target, reduction="sum"
            ).item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    loss /= len(data_loader.dataset)

    print(
        f"\n{dataset_name} set: Average loss: {loss:.4f}, Accuracy: {correct}/{len(data_loader.dataset)} ({100.0 * correct / len(data_loader.dataset):.0f}%)\n"
    )
    return loss

## Helper Code: Training Pipeline

For this assignment, the training pipeline is defined for you. Notice the similarities to the mathematical description of a trainer we saw above.

In [ ]:
@dataclass
class TrainingParameters:
    """Training parameters for a simple neural network trainer."""

    batch_size: int = 64
    test_batch_size: int = 1000
    epochs: int = 14
    lr: float = 1.0
    gamma: float = 0.7
    normalizer_mean: float = 0.1307
    normalizer_std: float = 0.3081
    no_cuda: bool = True  # Disable CUDA
    no_mps: bool = True  # Disable GPU on MacOS
    dry_run: bool = False
    seed: int = 1
    log_interval: int = 10
    save_model: bool = True


def configure_training_device(training_params):
    use_cuda = not training_params.no_cuda and torch.cuda.is_available()
    use_mps = not training_params.no_mps and torch.backends.mps.is_available()

    torch.manual_seed(training_params.seed)

    if use_cuda:
        device = torch.device("cuda")
    elif use_mps:
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    train_kwargs = {
        "batch_size": training_params.batch_size,
        "shuffle": True,
    }

    test_kwargs = {
        "batch_size": training_params.test_batch_size,
        "shuffle": False,
    }

    if use_cuda:
        cuda_kwargs = {
            "num_workers": 1,
            "pin_memory": True,
        }
        train_kwargs.update(cuda_kwargs)
        test_kwargs.update(cuda_kwargs)

    return device, train_kwargs, test_kwargs


def build_preprocessing_transform(training_params):
    transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                (training_params.normalizer_mean,),
                (training_params.normalizer_std,),
            ),
        ]
    )

    return transform


def build_data_loaders(train_kwargs, test_kwargs, transform):
    train_dataset = datasets.MNIST(
        "../data", train=True, download=True, transform=transform
    )
    test_dataset = datasets.MNIST(
        "../data", train=False, download=True, transform=transform
    )

    train_data, val_data = torch.utils.data.random_split(
        train_dataset, [0.9, 0.1], generator=torch.Generator().manual_seed(1)
    )
    train_loader = torch.utils.data.DataLoader(train_data, **train_kwargs)
    val_loader = torch.utils.data.DataLoader(val_data, **test_kwargs)
    test_loader = torch.utils.data.DataLoader(test_dataset, **test_kwargs)
    return train_loader, val_loader, test_loader


def train(training_params, device, train_loader, val_loader):
    model = Net().to(device)
    optimizer = optim.Adadelta(model.parameters(), lr=training_params.lr)
    scheduler = StepLR(optimizer, step_size=1, gamma=training_params.gamma)
    best_validation_loss = float("inf")

    for epoch in range(1, training_params.epochs + 1):
        run_training_epoch(
            training_params, model, device, train_loader, optimizer, epoch
        )
        validation_loss = evaluate(
            model, device, val_loader, dataset_name="Validation"
        )
        scheduler.step()

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            if training_params.save_model:
                torch.save(model.state_dict(), "mnist_model.pt")

## Example: Execute a Training Pipeline

With our training steps defined in modular fashion, we can easily define and execute a training pipeline.

Recall the training pipeline is defined by

$$
\Theta = g(\boldsymbol{\Omega}; \boldsymbol{X}, \boldsymbol{y}).
$$


In [ ]:
def execute_training_pipeline():
    training_params = TrainingParameters(
        epochs=1, dry_run=True, save_model=False
    )
    device, train_kwargs, test_kwargs = configure_training_device(
        training_params
    )
    transform = build_preprocessing_transform(training_params)
    train_loader, val_loader, _ = build_data_loaders(
        train_kwargs, test_kwargs, transform
    )
    train(training_params, device, train_loader, val_loader)


## Example: Execute an Evaluation Pipeline

With the training pipeline defined, we can run an evaluation pipeline on the test data. Save this pipeline for when you are ready to test your model.

The evaluation pipeline, $h$, accepts a set of model weights, along with a test set and test targets, and then produces a vector of performance metrics, $\boldsymbol{\mu}$.

$$
\boldsymbol{\mu} = h(\boldsymbol{\Theta}, \boldsymbol{X}_{\mathrm{TEST}}, \boldsymbol{y}_{\mathrm{TEST}})
$$

In [ ]:
def execute_evaluation_pipeline():
    training_params = TrainingParameters()

    device, train_kwargs, test_kwargs = configure_training_device(
        training_params
    )
    transform = build_preprocessing_transform(training_params)
    _, _, test_loader = build_data_loaders(
        train_kwargs, test_kwargs, transform
    )
    model = Net().to(device)
    model.load_state_dict(
        torch.load("mnist_model.pt", map_location=device, weights_only=True)
    )
    evaluate(model, device, test_loader)


## Task 01 - Part 2: Explore Width

Using the example above, define a neural network with a single hidden layer.

Modify the training pipeline to store the training and validation loss after each epoch.

Create a loop that iterates through several different hidden-layer widths. For each width:

* Create a new model.
* Train the model until convergence, or until the validation loss no longer improves for several epochs.
* Record the minimum validation loss achieved during training.

Plot the minimum validation loss with respect to the number of neurons in the hidden layer.

Use your results to describe how changing the width of the hidden layer affects the performance of the model.


In [ ]:
# Your code here

## Task 01 - Part 3: Explore Depth

Now explore the effect of **network depth**.

Using the example above, define several neural networks with increasing numbers of hidden layers.

Try to keep the other architectural choices approximately constant so that the primary variable being changed is the depth of the network.

Create a loop that iterates through several different network depths. For each depth:

* Create a new model.
* Train the model until convergence, or until the validation loss no longer improves for several epochs.
* Record the minimum validation loss achieved during training.

Plot the minimum validation loss with respect to the number of hidden layers.

Use your results to describe how changing the depth of the network affects model performance.


In [ ]:
# Your code here

# Task 01 - Part 4: Confounding Factors

As we have seen in class, a model can exhibit good performance for the wrong reason. Instead of learning the intended relationship between the input and output, it may learn to depend on a *spurious feature* that happens to be correlated with the target in the training data. This is called *shortcut learning*.

In this task, you will intentionally introduce a shortcut into MNIST and observe how it affects what is learned.

Choose a different pixel location for each digit class. During training, modify the pixel associated with the correct class so that its intensity carries information about the target label (e.g., make that pixel brighter for digits of that class).

Introduce a parameter, $\alpha$, which controls the strength of this feature. For example, you may modify the selected pixel value $x$ according to

$$
x' = (1-\alpha)x + \alpha,
$$

such that $\alpha = 0$ leaves the pixel unchanged and $\alpha = 1$ makes the confounding feature maximally strong.

Apply the artificial feature before image normalization so that $\alpha$ operates on pixel intensities in the range $[0,1]$.

Train several models using increasing values of $\alpha$. For each value:

* Create a new model.
* Apply the class-dependent confounding feature to the training data.
* Train the model until convergence.
* Record the training performance.
* Evaluate the model using the original, *unmodified* validation data.
* Evaluate the model using *counter-confounded* validation data that has been modified so that the bright pixel indicates the *incorrect* class.
* Record the validation performance.

Plot the training, unmodified validation, and *counter-confounded* validation performance with respect to the confounding-factor strength, $\alpha$.

Explain any findings from your results.

**This task is mandatory for graduate students and extra credit (+0.25 points on your final grade) for undergraduate students.**


In [ ]:
# Your code here

### Explanation

[your explanation here]

# Task 01 - Part 5: Final Evaluation

Now perform a final evaluation of your model using the MNIST test set. Based on the experiments you performed above, choose the architecture and training configuration that you believe performs best.

Using only the training and validation data to make your design decisions:

* Select your final network architecture.
* Select the training parameters you will use.
* Train a new instance of the model.
* Evaluate the best saved model checkpoint on the MNIST test set.
* Report the final test loss and classification accuracy.

Do not use the test set to make additional changes to the architecture or training process after evaluating the model.

Briefly compare the final test performance with the validation performance you observed during development. Discuss whether the test results are consistent with what you expected based on the validation data.

Note: Your grade does not depend on final performance. You will be graded on the experiments you perform, how well you explored the space of model architectures, and the insights you gained from them.


In [ ]:
# Your code here

### Explanation

[your explanation here]